# ETL da camada Raw para camada Silver - Microsoft Security Incident Prediction

Este notebook realiza o técnico ETL (Extract, Transform, Load) dos dados da camada Raw para a camada Silver. 
Ele processa o dataset Microsoft Security Incident Prediction, realizando transformações, limpeza de dados e salvando os dados processados.


## EXTRACT

Nesta seção, extraímos os dados do arquivo CSV da camada Raw.


In [26]:
# Carregador robusto usando Parquet (converte CSV em partes Parquet se necessário)
from pathlib import Path
import pandas as pd
import csv
import shutil

# Caminho relativo do notebook (transformer/) para o arquivo CSV (data_layer/raw/)
csv_file = Path('../data_layer/raw/GUIDE_Train.csv')
parquet_dir = csv_file.parent / 'parquet_parts'      # ../data_layer/raw/parquet_parts/
single_parquet = csv_file.with_suffix('.parquet')   # ../data_layer/raw/GUIDE_Train.parquet

if not csv_file.exists():
    raise FileNotFoundError(f"GUIDE_Train.csv não encontrado em {csv_file.absolute()}. Verifique onde o arquivo está.")

print("Usando CSV em:", csv_file.absolute())

# Se já existe um parquet único, carregue-o
if single_parquet.exists():
    print("Parquet único encontrado:", single_parquet)
    try:
        df = pd.read_parquet(single_parquet, engine='pyarrow')
        print("Carregado parquet único. shape:", df.shape)
    except Exception as e:
        print("Falha ao carregar parquet único:", type(e).__name__, e)
        raise

# Se já existem partes parquet em pasta, carregue (ou um sample)
elif parquet_dir.exists() and any(parquet_dir.glob('part_*.parquet')):
    parts = sorted(parquet_dir.glob('part_*.parquet'))
    print(f"Encontradas {len(parts)} partes parquet em {parquet_dir}. Tentando carregar todas (pode exigir RAM).")
    try:
        # tentativa de carregar tudo de uma vez
        df = pd.read_parquet(str(parquet_dir), engine='pyarrow')
        print("Carregado dataset Parquet (diretório). shape:", df.shape)
    except Exception as e:
        # fallback: carregar apenas a primeira parte para inspeção (use concat por partes se tiver RAM)
        print("Não foi possível carregar todo o dataset de uma vez:", type(e).__name__, e)
        print("Carregando apenas a primeira parte para inspeção.")
        df = pd.read_parquet(parts[0], engine='pyarrow')
        print("Carregado sample (parte 0). shape:", df.shape)
        print("Se quiser carregar tudo, concatene as partes manualmente ou aumente a memória disponível.")

# Caso não exista parquet, converte CSV para parquet por chunks
else:
    print("Nenhum parquet encontrado — convertendo CSV para parquet por chunks (streaming).")
    # sniff delimitador
    def sniff_delimiter(path, n_bytes=20000):
        try:
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                sample = f.read(n_bytes)
                dialect = csv.Sniffer().sniff(sample, delimiters=[',',';','\t','|'])
                return dialect.delimiter
        except Exception:
            return ','
    sep = sniff_delimiter(csv_file)
    print("Delimitador detectado:", repr(sep))

    # checa espaço em disco no diretório do parquet
    out_dir = parquet_dir
    out_dir.mkdir(exist_ok=True)
    usage = shutil.disk_usage(out_dir)
    print("Espaço livre no disco (destino):", f"{usage.free/1024/1024:.2f} MB")

    # parâmetros (ajuste chunksize conforme sua máquina)
    CHUNKSIZE = 200_000

    # leitura por chunks e escrita em parquet (pyarrow recomendado)
    it = pd.read_csv(csv_file, sep=sep, chunksize=CHUNKSIZE, iterator=True, engine='python', on_bad_lines='skip')
    i = 0
    for chunk in it:
        # downcast básico para reduzir uso de memória e tamanho do parquet
        for col, dtype in chunk.dtypes.items():
            if pd.api.types.is_integer_dtype(dtype):
                chunk[col] = pd.to_numeric(chunk[col], downcast='integer')
            elif pd.api.types.is_float_dtype(dtype):
                chunk[col] = pd.to_numeric(chunk[col], downcast='float')
            elif pd.api.types.is_object_dtype(dtype):
                # converte para category quando apropriado (cardinalidade baixa)
                try:
                    if chunk[col].nunique(dropna=True) / max(1, len(chunk)) < 0.5:
                        chunk[col] = chunk[col].astype('category')
                except Exception:
                    pass
        part_path = out_dir / f'part_{i:04d}.parquet'
        chunk.to_parquet(part_path, index=False, engine='pyarrow')
        print(f"Escreveu {part_path} (rows: {len(chunk)})")
        i += 1

    print("Conversão para parquet concluída. Partes escritas:", i)
    # opcional: criar um parquet único concatenando as partes (apenas se tiver RAM)
    try:
        parts = sorted(out_dir.glob('part_*.parquet'))
        if len(parts) == 1:
            # se só tiver uma parte, renomear para parquet único
            parts[0].rename(single_parquet)
            df = pd.read_parquet(single_parquet, engine='pyarrow')
            print("Único parquet gerado e carregado. shape:", df.shape)
        else:
            # tenta carregar todo o dataset via leitura de diretório (pyarrow dataset)
            try:
                df = pd.read_parquet(str(out_dir), engine='pyarrow')
                print("Carregado dataset parquet (diretório). shape:", df.shape)
            except Exception as e:
                print("Carregado apenas a primeira parte para inspeção:", e)
                df = pd.read_parquet(parts[0], engine='pyarrow')
                print("Sample carregado. shape:", df.shape)
                print("Se quiser o dataframe completo, concatene manualmente as partes ou aumente a memória.")
    except Exception as e:
        print("Erro pós-conversão ao tentar montar/ler parquet:", type(e).__name__, e)
        raise

# Ao final, 'df' estará definido como:
# - todo o dataset (se coube em memória),
# - ou um sample/primeira parte caso a máquina não tenha RAM suficiente.
print("Variável final disponível: df (shape pode variar conforme leitura).")


Usando CSV em: C:\Users\fabio\OneDrive\Área de Trabalho\SBD2-2025-2\transformer\..\data_layer\raw\GUIDE_Train.csv
Encontradas 48 partes parquet em ..\data_layer\raw\parquet_parts. Tentando carregar todas (pode exigir RAM).
Carregado dataset Parquet (diretório). shape: (9516837, 45)
Variável final disponível: df (shape pode variar conforme leitura).


## TRANSFORM

Nesta seção, realizamos as transformações necessárias nos dados.


### Padronização dos Nomes das Colunas

Padronizamos os nomes das colunas para facilitar o processamento. O padrão será: **todos os caracteres em minúsculo, separando palavras com `_`**.


In [27]:
# Backup dos nomes originais
original_columns = df.columns.tolist()

# Padronização dos nomes das colunas
df.columns = [col.lower().replace(' ', '_') for col in df.columns]

print("Primeiras 20 colunas padronizadas:")
for i, col in enumerate(df.columns[:20]):
    print(f"{i+1:2d}. {col}")

if len(df.columns) > 20:
    print(f"\n... e mais {len(df.columns) - 20} colunas")


Primeiras 20 colunas padronizadas:
 1. id
 2. orgid
 3. incidentid
 4. alertid
 5. timestamp
 6. detectorid
 7. alerttitle
 8. category
 9. mitretechniques
10. incidentgrade
11. actiongrouped
12. actiongranular
13. entitytype
14. evidencerole
15. deviceid
16. sha256
17. ipaddress
18. url
19. accountsid
20. accountupn

... e mais 25 colunas


### Análise de Valores Ausentes

Vamos analisar a presença de valores ausentes no dataset.


In [28]:
# Análise de valores nulos
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

# Criar DataFrame com informações de nulos
null_info = pd.DataFrame({
    'Coluna': null_counts.index,
    'Valores_Nulos': null_counts.values,
    'Percentual_Nulos': null_percentages.values
})

# Ordenar por percentual de nulos
null_info = null_info.sort_values('Percentual_Nulos', ascending=False)

print("Top 20 colunas com mais valores nulos:")
print(null_info.head(20).to_string(index=False))

# Identificar colunas com muitos valores ausentes (>50%)
high_missing_cols = null_info[null_info['Percentual_Nulos'] > 50]['Coluna'].tolist()
print(f"\nColunas com mais de 50% de valores ausentes: {len(high_missing_cols)}")
if high_missing_cols:
    print("Primeiras 10:", high_missing_cols[:10])


Top 20 colunas com mais valores nulos:
           Coluna  Valores_Nulos  Percentual_Nulos
     resourcetype        9509762         99.925658
    actiongrouped        9460773         99.410897
   actiongranular        9460773         99.410897
     threatfamily        9441956         99.213173
   emailclusterid        9420025         98.982729
antispamdirection        9339535         98.136965
            roles        9298686         97.707736
   suspicionlevel        8072708         84.825536
      lastverdict        7282572         76.523030
  mitretechniques        5468386         57.460120
    incidentgrade          51340          0.539465
        timestamp              0          0.000000
               id              0          0.000000
       entitytype              0          0.000000
       detectorid              0          0.000000
       alerttitle              0          0.000000
         category              0          0.000000
            orgid              0          0

### Remoção de Colunas com Muitos Valores Ausentes

Removemos colunas que têm mais de 80% de valores ausentes, pois não agregam valor significativo ao modelo.


In [29]:
# Definir threshold para remoção (80% de valores ausentes)
missing_threshold = 80

# Identificar colunas para remoção
cols_to_drop = null_info[null_info['Percentual_Nulos'] > missing_threshold]['Coluna'].tolist()

print(f"Colunas a serem removidas (>{missing_threshold}% ausentes): {len(cols_to_drop)}")
if cols_to_drop:
    print("Primeiras 10 colunas a serem removidas:")
    for i, col in enumerate(cols_to_drop[:10]):
        print(f"  {i+1}. {col}")

# Remover colunas com muitos valores ausentes
df_cleaned = df.drop(columns=cols_to_drop)
print(f"\nDimensões após remoção: {df_cleaned.shape}")
print(f"Colunas removidas: {df.shape[1] - df_cleaned.shape[1]}")

# Atualizar DataFrame principal
df = df_cleaned


Colunas a serem removidas (>80% ausentes): 8
Primeiras 10 colunas a serem removidas:
  1. resourcetype
  2. actiongrouped
  3. actiongranular
  4. threatfamily
  5. emailclusterid
  6. antispamdirection
  7. roles
  8. suspicionlevel

Dimensões após remoção: (9516837, 37)
Colunas removidas: 8


### Análise da Variável Target

Vamos analisar a variável target `HasDetections` para entender o balanceamento das classes.


In [30]:
# Análise da variável target
if 'hasdetections' in df.columns:
    target_counts = df['hasdetections'].value_counts()
    target_percentages = df['hasdetections'].value_counts(normalize=True) * 100
    
    print("Distribuição da variável target (HasDetections):")
    print(f"\nContagem:")
    for value, count in target_counts.items():
        print(f"  {value}: {count:,} ({target_percentages[value]:.2f}%)")
    
    # Verificar balanceamento
    balance_ratio = min(target_counts) / max(target_counts)
    print(f"\nRazão de balanceamento: {balance_ratio:.3f}")
    if balance_ratio < 0.1:
        print("  Dataset altamente desbalanceado!")
    elif balance_ratio < 0.3:
        print("  Dataset moderadamente desbalanceado")
    else:
        print(" Dataset relativamente balanceado")
        
    # Verificar valores ausentes na target
    target_missing = df['hasdetections'].isnull().sum()
    print(f"\nValores ausentes na variável target: {target_missing}")
    
else:
    print("Variável 'hasdetections' não encontrada no dataset.")
    print("Verificando colunas que podem ser a target:")
    potential_targets = [col for col in df.columns if 'detect' in col.lower() or 'target' in col.lower() or 'label' in col.lower()]
    if potential_targets:
        print(potential_targets)
    else:
        print("Nenhuma coluna target óbvia encontrada.")


Variável 'hasdetections' não encontrada no dataset.
Verificando colunas que podem ser a target:
['detectorid']


### Encoding de Variáveis Categóricas

Vamos aplicar Label Encoding para variáveis categóricas.


In [31]:
import sys
!{sys.executable} -m pip install scikit-learn

from sklearn.preprocessing import LabelEncoder

# Identificar colunas categóricas
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remover a variável target se ela for categórica
if 'hasdetections' in categorical_cols:
    categorical_cols.remove('hasdetections')

# Remover timestamp do encoding - ele deve ser convertido para datetime, não encoded
if 'timestamp' in categorical_cols:
    categorical_cols.remove('timestamp')
    print("Timestamp excluído do encoding - será convertido para datetime")

print(f"Colunas categóricas identificadas: {len(categorical_cols)}")

# Aplicar Label Encoding para colunas categóricas
label_encoders = {}
for col in categorical_cols:
    if col in df.columns:
        # Verificar número de categorias únicas
        unique_values = df[col].nunique()
        print(f"{col}: {unique_values} valores únicos")
        
        # Aplicar Label Encoding
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

print(f"\nEncoding aplicado em {len(label_encoders)} colunas categóricas.")


'C:\Users\fabio\OneDrive\µrea' nÆo ‚ reconhecido como um comando interno
ou externo, um programa oper vel ou um arquivo em lotes.


Timestamp excluído do encoding - será convertido para datetime
Colunas categóricas identificadas: 0

Encoding aplicado em 0 colunas categóricas.


### Tratamento de Valores Ausentes

Vamos tratar os valores ausentes restantes usando estratégias apropriadas.


In [32]:
# Verificar valores ausentes após encoding
missing_after_encoding = df.isnull().sum()
cols_with_missing = missing_after_encoding[missing_after_encoding > 0]

print("Valores ausentes após encoding:")
if len(cols_with_missing) > 0:
    print(cols_with_missing.sort_values(ascending=False))
    
    # Estratégias de tratamento
    for col in cols_with_missing.index:
        if df[col].dtype in ['int64', 'float64']:
            # Para colunas numéricas: imputação com mediana
            median_value = df[col].median()
            df[col] = df[col].fillna(median_value)
            print(f"  {col}: Preenchido com mediana ({median_value})")
        else:
            # Para colunas categóricas: imputação com moda
            mode_value = df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown'
            df[col] = df[col].fillna(mode_value)
            print(f"  {col}: Preenchido com moda ({mode_value})")
else:
    print("Nenhum valor ausente encontrado!")

# Verificação final
final_missing = df.isnull().sum().sum()
print(f"\nTotal de valores ausentes após tratamento: {final_missing}")


Valores ausentes após encoding:
lastverdict        7282572
mitretechniques    5468386
incidentgrade        51340
dtype: int64
  mitretechniques: Preenchido com moda (T1078;T1078.004)
  incidentgrade: Preenchido com moda (BenignPositive)
  lastverdict: Preenchido com moda (Suspicious)

Total de valores ausentes após tratamento: 0


## LOAD

Nesta seção, carregamos os dados processados para a camada Silver.


### Salvando os Dados Processados no Banco


Vamos salvar os dados processados na camada Silver.


In [33]:
import os
import re
import sys
import time
import psycopg2
from psycopg2 import sql
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import tempfile
import gc

print("--- Iniciando processo de carga OTIMIZADO (COPY) no PostgreSQL ---")
start_time = time.time()

# ---------- 1. Carregar DDL ----------
ddl_path = Path('../data_layer/silver/ddl.sql')
try:
    if not ddl_path.exists():
        raise FileNotFoundError(f"Arquivo DDL não encontrado em {ddl_path.absolute()}")
    ddl = ddl_path.read_text(encoding='utf-8')
    print(f"DDL carregado de: {ddl_path.absolute()}")
except Exception as e:
    print(f'Erro ao abrir arquivo ddl: {e}')
    sys.exit(1)

DB_SCHEMA = "silver"
TABLE_NAME = "microsoft_security_incident"

# ---------- 2. Ler parâmetros do docker-compose.yml ----------
def read_postgres_from_docker_compose(path):
    if not os.path.exists(path): return None
    try:
        with open(path, encoding='utf-8') as fh: text = fh.read()
        def find_var(name):
            m = re.search(rf'{re.escape(name)}\s*:\s*["\']?([^\n"\'#]+)', text)
            return m.group(1).strip() if m else None
        user = find_var('POSTGRES_USER') or 'postgres'
        password = find_var('POSTGRES_PASSWORD') or 'postgres'
        db = find_var('POSTGRES_DB') or 'microsoft-security'
        mport = re.search(r'ports\s*:\s*\n\s*-\s*["\']?(\d+):\d+', text, re.MULTILINE)
        port = mport.group(1) if mport else '5432'
        return dict(user=user, password=password, db=db, host='localhost', port=port)
    except: return None

current_dir = Path.cwd()
root_dir = current_dir.parent if current_dir.name == 'transformer' else current_dir
docker_compose_path = root_dir / 'docker-compose.yml'

pg = (read_postgres_from_docker_compose(str(docker_compose_path))
      or read_postgres_from_docker_compose(str(Path.cwd() / 'docker-compose.yml'))
      or {'user':'postgres','password':'postgres','db':'microsoft-security','host':'localhost','port':'5433'})

conn_info = f"host={pg['host']} dbname={pg['db']} user={pg['user']} password={pg['password']} port={pg['port']}"
print(f"Conectando em: {pg['host']}:{pg['port']} db={pg['db']}")

# ---------- 3. Preparar DataFrame ----------
try:
    if 'df' not in locals(): raise NameError
    print(f"Total de linhas no DF original: {len(df)}")
except NameError:
    print("Erro: variável 'df' não carregada.")
    sys.exit(1)

# Mapeamento e Seleção
column_mapping = {
    'orgid': 'org_id', 'incidentid': 'incident_id', 'alertid': 'alert_id',
    'detectorid': 'detector_id', 'alerttitle': 'alert_title', 'mitretechniques': 'mitre_techniques',
    'incidentgrade': 'incident_grade', 'entitytype': 'entity_type', 'evidencerole': 'evidence_role',
    'deviceid': 'device_id', 'accountsid': 'account_sid', 'accountupn': 'account_upn',
    'osfamily': 'os_family', 'osversion': 'os_version', 'countrycode': 'country_code', 'lastverdict': 'last_verdict'
}
df_db = df.rename(columns=column_mapping)
table_cols = ['id', 'org_id', 'incident_id', 'alert_id', 'timestamp', 'detector_id', 'alert_title', 'category', 'mitre_techniques', 'incident_grade', 'entity_type', 'evidence_role', 'device_id', 'sha256', 'url', 'account_sid', 'account_upn', 'os_family', 'os_version', 'country_code', 'state', 'city', 'last_verdict']
cols = [c for c in table_cols if c in df_db.columns]

# Criar cópia leve para inserção
df_insert = df_db[cols].copy()

# LIBERAR MEMÓRIA AGORA (CRÍTICO)
print("Liberando memória RAM do dataframe original...")
del df
del df_db
gc.collect()

# ---------- 4. Conversões ----------
print("Convertendo tipos...")
# Categorias -> Códigos
cat_cols = df_insert.select_dtypes(include=['category']).columns.tolist()
extra_cats = ['alert_title', 'country_code', 'state', 'city', 'category', 'mitre_techniques', 'incident_grade', 'entity_type', 'evidence_role', 'last_verdict', 'os_family', 'os_version']

mapping_data = {}
for col in set(cat_cols + extra_cats):
    if col in df_insert.columns:
        # Garante que é category sem usar função depreciada
        if df_insert[col].dtype.name != 'category':
            df_insert[col] = df_insert[col].astype('category')
        
        # Salva mapeamento
        mapping_data[col] = dict(enumerate(df_insert[col].cat.categories))
        
        # Converte para Inteiro (Int64 aceita Null)
        df_insert[col] = df_insert[col].cat.codes.replace(-1, np.nan).astype('Int64')

# Salvar mapeamento
try:
    with open('../data_layer/silver/category_mappings.txt', 'w', encoding='utf-8') as f:
        for c, m in mapping_data.items():
            f.write(f"\n[{c}]\n" + "\n".join([f" {k}: {v}" for k,v in list(m.items())[:10]]) + "\n")
except: pass

# Timestamp
if 'timestamp' in df_insert.columns:
    df_insert['timestamp'] = pd.to_datetime(df_insert['timestamp'], errors='coerce', utc=False)

# ---------- 5. Carga COPY (Correção Windows) ----------
print("\n=== INICIANDO CARGA VIA STREAMING (COPY) ===")
try:
    with psycopg2.connect(conn_info) as conn:
        conn.autocommit = True
        with conn.cursor() as cur:
            cur.execute(f"CREATE SCHEMA IF NOT EXISTS {DB_SCHEMA}")
            cur.execute(ddl) # Recria tabela com TEXT
            print("Tabela recriada (DDL OK).")

            # CRIAÇÃO DO ARQUIVO TEMPORÁRIO (CORREÇÃO WINDOWS)
            # newline='' impede que o Python adicione \r extra
            # lineterminator='\n' força o Pandas a usar apenas \n
            print("Gerando CSV temporário...")
            with tempfile.NamedTemporaryFile(mode='w+', suffix='.csv', delete=False, encoding='utf-8', newline='') as tmp:
                tmp_filename = tmp.name
                df_insert.to_csv(tmp, index=False, header=False, sep=';', na_rep='', lineterminator='\n')
            
            print(f"Enviando {len(df_insert)} linhas...")
            with open(tmp_filename, 'r', encoding='utf-8') as f:
                cols_sql = sql.SQL(', ').join(map(sql.Identifier, df_insert.columns))
                sql_copy = sql.SQL("COPY {}.{} ({}) FROM STDIN WITH (FORMAT CSV, DELIMITER ';', NULL '', QUOTE '\"')").format(
                    sql.Identifier(DB_SCHEMA), sql.Identifier(TABLE_NAME), cols_sql
                )
                cur.copy_expert(sql_copy, f)
            
            os.remove(tmp_filename)
            
            cur.execute(f"SELECT COUNT(*) FROM {DB_SCHEMA}.{TABLE_NAME}")
            print(f"\nSUCESSO! Linhas no banco: {cur.fetchone()[0]}")
            print(f"Tempo: {time.time() - start_time:.2f}s")

except Exception as e:
    print(f"\n ERRO: {e}")
    if 'tmp_filename' in locals(): os.remove(tmp_filename)

--- Iniciando processo de carga OTIMIZADO (COPY) no PostgreSQL ---
DDL carregado de: C:\Users\fabio\OneDrive\Área de Trabalho\SBD2-2025-2\transformer\..\data_layer\silver\ddl.sql
Conectando em: localhost:5433 db=microsoft-security
Total de linhas no DF original: 9516837
Liberando memória RAM do dataframe original...
Convertendo tipos...

=== INICIANDO CARGA VIA STREAMING (COPY) ===
Tabela recriada (DDL OK).
Gerando CSV temporário...
Enviando 9516837 linhas...

SUCESSO! Linhas no banco: 9516837
Tempo: 313.46s
